In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [22]:
from dotenv import load_dotenv
import os
from pathlib import Path
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
import json
from PIL import Image
import pandas as pd
from tqdm import tqdm

In [ ]:
load_dotenv()

True

In [63]:
DATA_DIR = Path(os.environ["DATA_DIR"])

IMAGE_DIR = DATA_DIR / "raw"
OUTPUT_CSV = DATA_DIR / "labels.csv"

IMAGE_EXTENSIONS = [".jpg"]

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded successfully.


In [ ]:
PROMPT = """
You are an expert marketing image analyst.

Analyze the image.

Choose exactly ONE mood:

- Energetic
- Calm
- Professional
- Playful

Choose exactly ONE content type:

- Product Showcase
- Lifestyle
- Testimonial
- Promotional

Return ONLY valid JSON.

{
    "mood": "Professional",
    "content_type": "Product Showcase"
}
"""

In [24]:
def analyze_image(image_path):
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(image_path),
                },
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = generated_ids[:, input_length:]
    # print(f"Input length: {input_length}, Generated IDs: {generated_ids.shape[1]}")

    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text

In [25]:
result  = analyze_image(IMAGE_DIR / "0_0a7ad22c5b040c00.jpg")
result = result[0].strip()
start = result.find("{")
end = result.rfind("}") + 1
print(json.loads(result[start:end]))

{'mood': 'Energetic', 'content_type': 'Promotional'}


In [26]:
jpg_files = sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(jpg_files)} images.\n")

Found 876 images.



In [ ]:
rows  = []
failed_files = []

for jpg_file in tqdm(jpg_files, desc="Processing images"):
    result = analyze_image(jpg_file)
    result = result[0].strip()
    start = result.find("{")
    end = result.rfind("}") + 1
    try:
        result_json = json.loads(result[start:end])
        rows.append({
            "image_name": jpg_file.name,
            "mood": result_json.get("mood", ""),
            "content_type": result_json.get("content_type", ""),
        })
    except json.JSONDecodeError:
        failed_files.append(jpg_file)
        print(f"Failed to parse JSON for {jpg_file.name}: {result}")
    

df = pd.DataFrame(rows)
print(f"Saved {len(df)} rows")
print(f"Failed to process {len(failed_files)} files")

Processing images:  28%|██▊       | 241/876 [10:33<28:25,  2.69s/it]

Failed to parse JSON for 0_edc7696dbe9478d8.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images:  51%|█████     | 448/876 [19:38<17:51,  2.50s/it]

Failed to parse JSON for 2_1dce19795dac8c20.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images:  56%|█████▌    | 487/876 [21:22<18:23,  2.84s/it]

...

Processing images:  88%|████████▊ | 773/876 [34:11<04:39,  2.72s/it]

Failed to parse JSON for 3_48312460bff0993c.jpg: !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


Processing images: 100%|██████████| 876/876 [38:52<00:00,  2.66s/it]

Saved 870 rows
Failed to process 6 files


,image_name,mood,content_type
0,0_005617151e189073.jpg,Professional,Product Showcase
1,0_00e26c825cbbf94f.jpg,Energetic,Product Showcase
2,0_010383a788e0ea6a.jpg,Professional,Product Showcase
3,0_010c611f83122d42.jpg,Calm,Product Showcase
4,0_014c34acf67c53b5.jpg,Professional,Promotional


In [34]:
for failed_file in failed_files:
    print(f"Failed file: {failed_file.name}")
    doc = {"image_name": failed_file.name, "mood": "", "content_type": ""}
    df_failed = pd.DataFrame([doc])
    df = pd.concat([df, df_failed], ignore_index=True) 

Failed file: 0_edc7696dbe9478d8.jpg
Failed file: 2_1dce19795dac8c20.jpg
Failed file: 2_3882c24f5e9adf2a.jpg
Failed file: 2_56f358bac323b688.jpg
Failed file: 2_5c87d76524694c78.jpg
Failed file: 3_48312460bff0993c.jpg


In [52]:
filename_to_fix = '0_edc7696dbe9478d8.jpg' 

df.loc[df['image_name'] == filename_to_fix, ['mood', 'content_type']] = ['Calm', 'Lifestyle']
print(df[df['image_name'] == filename_to_fix])

                 image_name  mood content_type
870  0_edc7696dbe9478d8.jpg  Calm    Lifestyle


In [62]:
df["mood"].value_counts()

,count
mood,
Calm,333
Professional,293
Energetic,171
Playful,79


In [61]:
df["content_type"].value_counts()

,count
content_type,
Lifestyle,443
Promotional,298
Product Showcase,129
Testimonial,6


In [57]:
valid_types = ['Lifestyle', 'Promotional', 'Product Showcase', 'Testimonial']
invalid_rows = df[~df['content_type'].isin(valid_types)]
print(invalid_rows[['image_name', 'content_type']])

                 image_name  content_type
197  0_d066d8c21f939429.jpg        Sports
343  1_89d9552a610e6a67.jpg        Sports
871  2_1dce19795dac8c20.jpg  Professional


In [58]:
df.loc[df['content_type'] == 'Sports', 'content_type'] = 'Lifestyle'
df.loc[df['content_type'] == 'Professional', 'content_type'] = 'Promotional'

In [56]:
empty_rows = df[df['mood'].isna() | (df['mood'] == '') | (df['mood'] == 'Unknown')]
print(empty_rows[['image_name', 'mood', 'content_type']])

Empty DataFrame
Columns: [image_name, mood, content_type]
Index: []


In [64]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"{len(df)} rows saved")
df.head() 

876 rows saved


,image_name,mood,content_type
0,0_005617151e189073.jpg,Professional,Product Showcase
1,0_00e26c825cbbf94f.jpg,Energetic,Product Showcase
2,0_010383a788e0ea6a.jpg,Professional,Product Showcase
3,0_010c611f83122d42.jpg,Calm,Product Showcase
4,0_014c34acf67c53b5.jpg,Professional,Promotional
